# PCT Training - Occlusion

Trains Menghao PCT model from the Point-Transformers implementation: https://github.com/qq456cvb/Point-Transformers

Changes: Added an occlusion function that randomly drops between 0.1 to 0.9 fraction of points from the cloud in a plane-based manner

Data: fullmodelnet40. This is a bit awkward for testing, because the training happens on (heavily) occluded clouds, but the testing is done on full clouds. Could alternatively use a different dataset to test on, such as partialmodelnet40.

## Env prep

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# Paths, folders
import os

REPO_PATH = '/content/pointcloud-bench'
DRIVE_PATH = '/content/drive/MyDrive/pointcloud-bench'
results_dir = os.path.join(DRIVE_PATH, 'results')
os.makedirs(results_dir, exist_ok=True)

In [ ]:
# Get repo
!git clone --recurse-submodules --branch pct-plots https://github.com/DavidClaszen/pointcloud-bench {REPO_PATH}

# Submodule handling
%cd {REPO_PATH}
!git submodule update --init --recursive
%cd repos/Point-Transformers
!git fetch origin pct-occlusion
!git checkout pct-occlusion
!git pull origin pct-occlusion
%cd {REPO_PATH}

%pip install -r envs/pct/requirements.txt

In [ ]:
# Check for CUDA/GPU
import torch, sys
print(sys.version)
print('Torch:', torch.__version__, 'CUDA:', torch.version.cuda, 'GPU:', torch.cuda.is_available())

In [ ]:
# Copy and unzip only the fullmodelnet40 set
# Evaluation will be done in other notebook
!rsync -avP {DRIVE_PATH}/datasets/fullmodelnet40.tar.gz {REPO_PATH}/datasets
%cd {REPO_PATH}
!tar -xvzf datasets/fullmodelnet40.tar.gz -C datasets

# Model Training

Since we're only using PAPNet style data here, always set `use_papnet_loader` to `True`.

New arguments:
- occlusion: bool
- occlusion_min: float, default 0.1
- occlusion_max: float, default 0.9


In [ ]:
%cd /content/pointcloud-bench/repos/Point-Transformers
!python train_cls.py --help

In [ ]:
# Train Menghao
!python train_cls.py model=Menghao use_papnet_loader=True batch_size=512 learning_rate=0.0005 epoch=50 workers=4 step_size=15 data_path=../../datasets/fullmodelnet40/ occlusion=True occlusion_min=0.1 occlusion_max=0.9

In [ ]:
# Zip Point-Transformers logs, included last best model
!zip -r results.zip ./log/cls/Menghao/